<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 4.1：FIRRTL 简介

**上一步：[生成器：类型](3.6_types.ipynb)**<br>
**下一步：[FIRRTL AST 遍历](4.2_firrtl_ast_traversal.ipynb)**

## 动机
您已经学习了一些 Scala 并编写了一些 Chisel，对于 90% 的用户来说，这足以成为 Chisel 的爱好者。

然而，某些用例更适合表示为 Chisel 设计的程序化转换，而不是生成器。

例如，假设我们要计算设计中寄存器的数量。这很难作为生成器来完成，因此，我们可以编写一个 FIRRTL 过程来为我们完成它。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.iotesters.{ChiselFlatSpec, Driver, PeekPokeTester}
import firrtl._

## 什么是 FIRRTL？
您可能已经意识到，当您执行 Chisel 设计时，它会进行细化（执行周围的 Scala 代码）以构造生成器的实例，并解析所有 Scala 参数。

Chisel 不会直接生成 Verilog，而是生成一种称为 FIRRTL 的中间表示，它表示已细化（参数已解析）的 RTL 实例。它可以被序列化（转换为字符串以写入文件），并且这种序列化语法是人类可读的。然而，在内部，它并不是表示为一个长字符串。相反，它是一个组织为节点树的数据结构，称为抽象语法树 (AST)。

让我们来看看！我们将采用一个简单的 Chisel 设计，对其进行细化，并检查它生成的 FIRRTL！

首先，我们定义一个 Chisel 模块，它将其输入信号延迟两个周期。

In [ ]:
class DelayBy2(width: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  val r0 = RegNext(io.in)
  val r1 = RegNext(r0)
  io.out := r1
}

接下来，让我们对其进行细化、序列化并打印出它生成的 FIRRTL。

In [ ]:
println(chisel3.Driver.emit(() => new DelayBy2(4)))

如您所见，序列化的 FIRRTL 与我们的 Chisel 设计非常相似，所有生成器参数都已解析。

## FIRRTL AST

如前所述，FIRRTL 表示可以序列化为字符串，但在内部，它是一个称为 AST（抽象语法树）的数据结构。此数据结构是一个节点树，其中一个节点可以包含子节点。此数据结构中没有循环。

让我们看看内部数据结构是什么样的：

In [ ]:
val firrtlSerialization = chisel3.Driver.emit(() => new DelayBy2(4))
val firrtlAST = firrtl.Parser.parse(firrtlSerialization.split("\n").toIterator, Parser.GenInfo("file.fir"))

println(firrtlAST)

显然，数据结构的序列化并不那么美观，但您可以看到一些内部表示 RTL 设计的类等。让我们尝试美化一下，使其易于理解。

In [ ]:
println(stringifyAST(firrtlAST))

这是保存 FIRRTL AST 的内部数据结构。它是一个树形结构，其根节点是 **Circuit**，它有 3 个子节点：**@[file.fir@2.0]**、**ArrayBuffer** 和 **cmd5WrapperHelperDelayBy2**。以下是序列化的 `Circuit` 实际 Scala 类的定义：<a name="circuit"></a><img src="images/circuit.png" alt="Circuit case class" />



如您所见，它有三个子节点：`info: Info`、`Modules: Seq[DefModule]` 和 `main: String`。它扩展了 `FirrtlNode`，所有 FIRRTL AST 节点都必须这样做。暂时忽略 `def mapXXXX` 函数。

许多 FIRRTL 节点包含一个 `info: Info` 字段，解析器可以在其中插入文件信息（如行号和列号），或者插入一个 `NoInfo` 标记。在此示例中，**@[file.fir@2.0]** 将引用 FIRRTL 文件、第 2 行、第 0 列。

下一节将详细概述所有这些 FIRRTL 节点。

# FIRRTL 节点描述

本节描述了在 [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/ucb-bar/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala) 中找到的常见 FirrtlNode。

有关此处未提及的组件的更多详细信息，请参阅 [FIRRTL 规范](https://github.com/ucb-bar/firrtl/blob/master/spec/spec.pdf)。


## 电路
电路是任何 Firrtl 数据结构的根节点。只有一个电路，该电路包含模块定义列表和顶层模块的名称。

#### FirrtlNode 声明
```scala 
Circuit(info: Info, modules: Seq[DefModule], main: String)
```

#### 具体语法
```
circuit Adder:
  ... //模块列表
```
#### 内存中表示
```scala
Circuit(NoInfo, Seq(...), "Adder")
```

## 模块

模块是 Firrtl 中的模块化单元，从不直接嵌套（声明模块实例有其自己的具体语法和 AST 表示）。每个模块都有一个名称、端口列表和包含其实现的模块体。

#### FirrtlNode 声明
```scala
Module(info: Info, name: String, ports: Seq[Port], body: Stmt) extends DefModule
```

#### 具体语法
```
module Adder:
  ... // 端口列表
  ... // 语句
```
#### 内存中表示
```scala
Module(NoInfo, "Adder", Seq(...), )
```

## 端口
端口定义模块 IO 的一部分，并具有名称、方向（输入或输出）和类型。

#### FirrtlNode 声明
```scala
class Port(info: Info, name: String, direction: Direction, tpe: Type)
```
#### 具体语法
```
input x: UInt
```

#### 内存中表示
```scala
Port(NoInfo, "x", INPUT, UIntType(UnknownWidth))
```

## 语句
语句用于描述模块内的组件以及它们如何交互。以下是一些常用的语句：

### 语句块
一组语句。通常用作模块声明中的主体字段。

### Wire 声明
Wire 声明，包含名称和类型。它可以同时作为源（连接*自*）和宿（连接*至*）。
#### FirrtlNode 声明
```scala
DefWire(info: Info, name: String, tpe: Type)
```
#### 具体语法
```
wire w: UInt
```
#### 内存中表示
```scala
DefWire(NoInfo, "w", UIntType(UnknownWidth))
```

### 寄存器声明
寄存器声明，包含名称、类型、时钟信号、复位信号和复位值。
#### FirrtlNode 声明
```scala
DefRegister(info: Info, name: String, tpe: Type, clock: Expression, reset: Expression, init: Expression)
```

### 连接
表示从源到宿的有向连接。请注意，它遵循最后连接语义，如 Chisel 中所述。

#### FirrtlNode 声明
```scala
Connect(info: Info, loc: Expression, expr: Expression)
```

### 其他语句
其他语句类型，如 `DefMemory`、`DefNode`、`IsInvalid`、`Conditionally` 等在此处省略；有关更多详细信息，请参阅 [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/freechipsproject/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala)。

## 表达式
表达式表示对已声明组件的引用或逻辑和算术运算。以下是一些常用的表达式：

### 引用
对已声明组件（例如连线、寄存器或端口）的引用。它具有名称和类型字段。请注意，它不包含指向实际声明的指针，而仅包含名称作为字符串。

#### FirrtlNode 声明
```scala
Reference(name: String, tpe: Type)
```

### DoPrim
匿名原语操作，例如 `Add`、`Sub` 或 `And`、`Or` 或子字选择 (`Bits`)。操作类型由 `op: PrimOp` 字段指示。请注意，所需参数和常量的数量由 `op` 决定。

#### FirrtlNode 声明
```scala
DoPrim(op: PrimOp, args: Seq[Expression], consts: Seq[BigInt], tpe: Type)
```

### 其他表达式
其他表达式，包括 `SubField`、`SubIndex`、`SubAccess`、`Mux`、`ValidIf` 等，在 [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/freechipsproject/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala) 和 [FIRRTL 规范](https://github.com/ucb-bar/firrtl/blob/master/spec/spec.pdf) 中有更详细的描述。

# 回到我们的示例

让我们再看一下示例中的 FIRRTL AST。希望设计的结构更有意义！

In [ ]:
println(stringifyAST(firrtlAST))

本节到此结束！在下一节中，我们将了解 FIRRTL 转换如何遍历此 AST 并对其进行修改。